In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
infamouscoder_depression_reddit_cleaned_path = kagglehub.dataset_download('infamouscoder/depression-reddit-cleaned')

print('Data source import complete.')


100%|██████████| 979k/979k [00:00<00:00, 61.5MB/s]

Extracting files...
Data source import complete.


In [2]:
import numpy as np
import pandas as pd
import re
import os

import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout


In [3]:
df = pd.read_csv(
    "/root/.cache/kagglehub/datasets/infamouscoder/depression-reddit-cleaned/versions/1/depression_dataset_reddit_cleaned.csv"
)

print(df.shape)
df.head()


(7731, 2)


,clean_text,is_depression
0,we understand that most people who reply immed...,1
1,welcome to r depression s check in post a plac...,1
2,anyone else instead of sleeping more when depr...,1
3,i ve kind of stuffed around a lot in my life d...,1
4,sleep is my greatest and most comforting escap...,1


In [4]:
print(df['is_depression'].value_counts())


is_depression
0    3900
1    3831
Name: count, dtype: int64


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text


In [6]:
df['clean_text'] = df['clean_text'].apply(clean_text)


In [7]:
X = df['clean_text']
y = df['is_depression']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [8]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [9]:
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)


LinearSVC()

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
rf_model.fit(X_train_tfidf, y_train)


RandomForestClassifier(n_estimators=200, random_state=42)

In [11]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)


In [12]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/5
87/87 ━━━━━━━━━━━━━━━━━━━━ 44s 453ms/step - accuracy: 0.7736 - loss: 0.4863 - val_accuracy: 0.9305 - val_loss: 0.1613
Epoch 2/5
87/87 ━━━━━━━━━━━━━━━━━━━━ 32s 365ms/step - accuracy: 0.9626 - loss: 0.1076 - val_accuracy: 0.9580 - val_loss: 0.1176
Epoch 3/5
87/87 ━━━━━━━━━━━━━━━━━━━━ 36s 414ms/step - accuracy: 0.9847 - loss: 0.0515 - val_accuracy: 0.9645 - val_loss: 0.0999
Epoch 4/5
87/87 ━━━━━━━━━━━━━━━━━━━━ 34s 332ms/step - accuracy: 0.9901 - loss: 0.0328 - val_accuracy: 0.9677 - val_loss: 0.1133
Epoch 5/5
87/87 ━━━━━━━━━━━━━━━━━━━━ 36s 268ms/step - accuracy: 0.9934 - loss: 0.0238 - val_accuracy: 0.9645 - val_loss: 0.1184


In [14]:
print("SVM Accuracy:", accuracy_score(y_test, svm_model.predict(X_test_tfidf)))
print("RF Accuracy:", accuracy_score(y_test, rf_model.predict(X_test_tfidf)))


SVM Accuracy: 0.9663865546218487
RF Accuracy: 0.9689722042663219


In [15]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


In [16]:
def predict_depression(text):
    original_text = text
    text = clean_text(text)

    # TF-IDF
    tfidf_text = tfidf.transform([text])

    # SVM probability (via sigmoid)
    svm_score = svm_model.decision_function(tfidf_text)[0]
    svm_prob = sigmoid(svm_score)

    # RF probability
    rf_prob = rf_model.predict_proba(tfidf_text)[0][1]

    # BiLSTM probability
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    lstm_prob = model.predict(pad, verbose=0)[0][0]

    # Weighted ensemble
    final_prob = (
        0.3 * svm_prob +
        0.1 * rf_prob +
        0.6 * lstm_prob
    )

    prediction = "Depressed" if final_prob > 0.33 else "Not Depressed"

    # Ethical safeguard
    high_risk_keywords = [
        "hopeless", "worthless", "empty", "tired of life",
        "no reason to live", "give up", "hate myself",
        "alone", "numb", "drained"
    ]

    if final_prob > 0.25 and any(w in original_text.lower() for w in high_risk_keywords):
        prediction = "Depressed (High-Risk Override)"

    print(f"SVM: {svm_prob:.2f}, RF: {rf_prob:.2f}, BiLSTM: {lstm_prob:.2f}")
    print(f"Final Depression Probability: {final_prob:.2f}")

    return prediction


In [17]:
sample_text = "I feel completely drained and hopeless"
print("Prediction:", predict_depression(sample_text))


SVM: 0.40, RF: 0.08, BiLSTM: 0.37
Final Depression Probability: 0.35
Prediction: Depressed (High-Risk Override)


In [18]:
os.makedirs("saved_models", exist_ok=True)


In [19]:
joblib.dump(tfidf, "saved_models/tfidf_vectorizer.pkl")


['saved_models/tfidf_vectorizer.pkl']

In [20]:
joblib.dump(svm_model, "saved_models/svm_model.pkl")


['saved_models/svm_model.pkl']

In [21]:
joblib.dump(rf_model, "saved_models/rf_model.pkl")


['saved_models/rf_model.pkl']

In [22]:
joblib.dump(tokenizer, "saved_models/tokenizer.pkl")


['saved_models/tokenizer.pkl']

In [23]:
model.save("saved_models/bilstm_model.keras")


In [24]:
ensemble_config = {
    "svm_weight": 0.3,
    "rf_weight": 0.1,
    "lstm_weight": 0.6,
    "threshold": 0.33,
    "high_risk_threshold": 0.25,
    "high_risk_keywords": [
        "hopeless", "worthless", "empty", "tired of life",
        "no reason to live", "give up", "hate myself",
        "alone", "numb", "drained"
    ],
    "max_len": max_len
}

joblib.dump(ensemble_config, "saved_models/ensemble_config.pkl")


['saved_models/ensemble_config.pkl']

In [25]:
import shutil

shutil.make_archive(
    base_name="saved_models",
    format="zip",
    root_dir="saved_models"
)


'/content/saved_models.zip'

In [26]:
from google.colab import files

files.download("saved_models.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>